Used Vehicle Market Analysis
Introduction

The purpose of this project is to explore used vehicle advertisements and identify the factors that influence vehicle prices. The results of the analysis will also be used to build an interactive Streamlit application.

Project steps:
1. Load and inspect the dataset.
2. Examine missing values, duplicates, and data types.
3. Process missing and unusual values.
4. Analyze the distributions of vehicle prices, mileage, and other characteristics.
5. Study the relationships between vehicle characteristics and price.
6. Create visualizations and summarize the main findings.


In [1]:
import pandas as pd
import plotly.express as px
df = pd.read_csv("../vehicles_us.csv")

In [2]:
df.head()
df.shape
df.columns
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


In [3]:
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False))

display(df.describe())



Missing values:


is_4wd          25953
paint_color      9267
odometer         7892
cylinders        5260
model_year       3619
condition           0
model               0
price               0
fuel                0
type                0
transmission        0
date_posted         0
days_listed         0
dtype: int64

,price,model_year,cylinders,odometer,is_4wd,days_listed
count,51525.000000,47906.000000,46265.000000,43633.000000,25572.0,51525.00000
mean,12132.464920,2009.750470,6.125235,115553.461738,1.0,39.55476
std,10040.803015,6.282065,1.660360,65094.611341,0.0,28.20427
min,1.000000,1908.000000,3.000000,0.000000,1.0,0.00000
25%,5000.000000,2006.000000,4.000000,70000.000000,1.0,19.00000
50%,9000.000000,2011.000000,6.000000,113000.000000,1.0,33.00000
75%,16839.000000,2014.000000,8.000000,155000.000000,1.0,53.00000
max,375000.000000,2019.000000,12.000000,990000.000000,1.0,271.00000


## Processing missing values

Missing values are restored where possible instead of removing rows, since deleting observations may lead to the loss of useful information.

In [4]:
# Missing is_4wd values are interpreted as vehicles without four-wheel drive
df['is_4wd'] = df['is_4wd'].fillna(0).astype(bool)


# Fill missing cylinder values with the median for each vehicle type
cylinders_median = (
    df.groupby('type')['cylinders']
    .transform('median')
)

df['cylinders'] = (
    df['cylinders']
    .fillna(cylinders_median)
    .fillna(df['cylinders'].median())
    .round()
    .astype('int')
)


# Fill missing model years with the median year for each vehicle model
model_year_median = (
    df.groupby('model')['model_year']
    .transform('median')
)

df['model_year'] = (
    df['model_year']
    .fillna(model_year_median)
    .fillna(df['model_year'].median())
    .round()
    .astype('int')
)


# Fill missing odometer values using the median
# for vehicles with the same model year and condition
odometer_median = (
    df.groupby(['model_year', 'condition'])['odometer']
    .transform('median')
)

df['odometer'] = (
    df['odometer']
    .fillna(odometer_median)
    .fillna(df['odometer'].median())
)

In [5]:
print("Missing values after processing:")
display(df.isna().sum().sort_values(ascending=False))

print(df['is_4wd'].dtype)

Missing values after processing:


paint_color     9267
model_year         0
model              0
condition          0
price              0
cylinders          0
fuel               0
transmission       0
odometer           0
type               0
is_4wd             0
date_posted        0
days_listed        0
dtype: int64

bool


In [6]:
check_cols = ['is_4wd', 'cylinders', 'model_year', 'odometer']

print('Missing values:')
display(df[check_cols].isna().sum())

print('Data types:')
display(df[check_cols].dtypes)

print('Summary statistics:')
display(df[check_cols].describe())

Missing values:


is_4wd        0
cylinders     0
model_year    0
odometer      0
dtype: int64

Data types:


is_4wd           bool
cylinders       int64
model_year      int64
odometer      float64
dtype: object

Summary statistics:


,cylinders,model_year,odometer
count,51525.000000,51525.000000,51525.000000
mean,6.130810,2009.793557,115237.452217
std,1.658414,6.099381,62202.349431
min,3.000000,1908.000000,0.000000
25%,4.000000,2007.000000,73179.000000
50%,6.000000,2011.000000,114773.000000
75%,8.000000,2014.000000,151752.000000
max,12.000000,2019.000000,990000.000000


In [11]:
print(
    df[['cylinders', 'model_year', 'odometer']]
    .agg(['min', 'median', 'max'])
)

        cylinders  model_year  odometer
min           3.0      1908.0       0.0
median        6.0      2011.0  114773.0
max          12.0      2019.0  990000.0


In [7]:
fig = px.histogram(
    df,
    x="price",
    nbins=50,
    title="Distribution of Vehicle Prices"
)

fig.show()

In [8]:
fig = px.histogram(
    df,
    x="odometer",
    nbins=50,
    title="Distribution of Odometer Readings"
)

fig.show()

In [9]:
fig = px.scatter(
    df,
    x="odometer",
    y="price",
    title="Vehicle Price vs Odometer"
)

fig.show()

In [10]:
fig = px.scatter(
    df,
    x="model_year",
    y="price",
    title="Vehicle Price vs Model Year"
)

fig.show()
